In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchaudio")

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torchaudio import datasets, transforms

In [3]:
LABELS = ['backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five',
          'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn',
          'left', 'marvin', 'nine', 'no', 'off', 'on', 'one', 'right',
          'seven', 'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up',
          'visual', 'wow', 'yes', 'zero']

labelToIdx = {l: i for i, l in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)  # 35

In [4]:
mfcc_layer = torchaudio.transforms.MFCC(
    sample_rate=16000,
    n_mfcc=40,
    melkwargs={
        "n_fft": 512,
        "hop_length": 160,
        "win_length": 400,
        "n_mels": 64,
        "f_min": 0.0,
        "f_max": 8000.0,
    }
)

In [5]:
TARGET_LENGTH = 16000

def collate_fn(batch, training=False):
    waveforms, labels = [], []
    for waveform, sample_rate, label, *_ in batch:
        # Pad or trim to exactly 1 second
        if waveform.shape[-1] < TARGET_LENGTH:
            waveform = F.pad(waveform, (0, TARGET_LENGTH - waveform.shape[-1]))
        else:
            waveform = waveform[..., :TARGET_LENGTH]
        waveforms.append(waveform)
        labels.append(labelToIdx[label])
    return torch.stack(waveforms), torch.tensor(labels)

train_collate = lambda batch: collate_fn(batch, training=True)
test_collate  = lambda batch: collate_fn(batch, training=False)

In [6]:
SC_train = datasets.SPEECHCOMMANDS(root='data', subset='training', download=True, url='speech_commands_v0.02')
SC_test  = datasets.SPEECHCOMMANDS(root='data', subset='testing',  download=True, url='speech_commands_v0.02')

In [7]:

train_loader = torch.utils.data.DataLoader(SC_train, batch_size=64,   num_workers=4, pin_memory=True, shuffle=True,  collate_fn=train_collate)
test_loader  = torch.utils.data.DataLoader(SC_test,  batch_size=1024, num_workers=4, pin_memory=True, shuffle=False, collate_fn=test_collate)

In [13]:
class KeywordGRU(nn.Module):
    def __init__(self, n_mfcc=40, hidden_size=128, num_layers=2, num_classes=NUM_CLASSES):
        super().__init__()
        self.mfcc = mfcc_layer
        self.gru = nn.GRU(
            input_size=n_mfcc,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.5,
            bidirectional=False
        )
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.mfcc(x)               # (batch, 1, n_mfcc, time)
        x = x.squeeze(1)               # (batch, n_mfcc, time)
        x = x.permute(0, 2, 1)         # (batch, time, n_mfcc)
        out, _ = self.gru(x)
        out = out.mean(dim=1)          # mean pool over time
        return self.classifier(out)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KeywordGRU().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
criterion = nn.CrossEntropyLoss()

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total model parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Device: cuda
GPU Name: NVIDIA H200 MIG 2g.35gb
GPU Memory: 34.90 GB
Total model parameters: 436,003
Trainable parameters: 436,003


In [15]:
def train(epoch=None):
    model.train()
    total_loss = 0
    correct = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (output.argmax(dim=1) == target).sum().item()

        if batch_idx % 500 == 0 and epoch is not None:
            print(f"Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}")

    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / len(train_loader.dataset)
    if epoch is not None:
        print(f"Epoch {epoch} - Avg Loss: {avg_loss:.6f} | Train Acc: {accuracy:.2f}%")
    return accuracy

In [16]:
def test():
  model.eval()
  test_loss = 0
  correct = 0
  with torch.no_grad():
    for data, target in test_loader:
      data, target = data.to(device), target.to(device)
      output = model(data)
      test_loss += criterion(output, target).item() 
      pred = output.argmax(dim=1, keepdim=True)
      correct += pred.eq(target.view_as(pred)).sum().item()
      
  test_loss /= len(test_loader.dataset)
  accuracy = 100. * correct / len(test_loader.dataset)

  print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
      test_loss, correct, len(test_loader.dataset),
      100. * correct / len(test_loader.dataset)))
  return accuracy  

In [ ]:
for epoch in range(1, 21):
    train_acc = train(epoch)
    val_acc = test()
    scheduler.step(val_acc)

Epoch 1 [0/84843] Loss: 3.580590
Epoch 1 [32000/84843] Loss: 0.738598
Epoch 1 [64000/84843] Loss: 0.316194
Epoch 1 - Avg Loss: 0.947037 | Train Acc: 73.47%

Test set: Average loss: 0.0005, Accuracy: 9309/11005 (85%)

Epoch 2 [0/84843] Loss: 0.500891
Epoch 2 [32000/84843] Loss: 0.465743
Epoch 2 [64000/84843] Loss: 0.394372
Epoch 2 - Avg Loss: 0.435229 | Train Acc: 87.11%

Test set: Average loss: 0.0004, Accuracy: 9543/11005 (87%)

Epoch 3 [0/84843] Loss: 0.390446
Epoch 3 [32000/84843] Loss: 0.298858
Epoch 3 [64000/84843] Loss: 0.335057
Epoch 3 - Avg Loss: 0.347248 | Train Acc: 89.69%

Test set: Average loss: 0.0004, Accuracy: 9733/11005 (88%)

Epoch 4 [0/84843] Loss: 0.324810
Epoch 4 [32000/84843] Loss: 0.334345
Epoch 4 [64000/84843] Loss: 0.293020
Epoch 4 - Avg Loss: 0.299246 | Train Acc: 90.95%

Test set: Average loss: 0.0004, Accuracy: 9805/11005 (89%)

Epoch 5 [0/84843] Loss: 0.303821
Epoch 5 [32000/84843] Loss: 0.306980
Epoch 5 [64000/84843] Loss: 0.357566
Epoch 5 - Avg Loss: 0.268